In [0]:

from pyspark.sql.functions import col, from_json
import uuid

bootstrap_server = "pkc-619z3.us-east1.gcp.confluent.cloud:9092"
api_key = "ZWSEEXNPTALGFSAS"
api_secret = "cfltHleM7FMu1XNTdcfRK34pwFF8GbCoO/JGkWCevJXYqCjp5MMhJPvsXp62e+tw"
topic_name = "loan_transaction"

random_id = uuid.uuid4().hex[:6]

checkpoint_path_final = f"/Volumes/workspace/default/data_csv/checkpoints/cp_final_{random_id}"
bronze_path = "/Volumes/workspace/default/data_csv/loan_bronze"

# df_kafka_raw = spark.readStream \
#     .format("kafka") \
#     .option("kafka.bootstrap.servers", bootstrap_server) \
#     .option("kafka.security.protocol", "SASL_SSL") \
#     .option("kafka.sasl.mechanism", "PLAIN") \
#     .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";') \
#     .option("subscribe", "loan_transaction") \
#     .option("startingOffsets", "earliest") \
#     .option("kafka.group.id", f"group_{random_id}") \
#     .option("kafka.session.timeout.ms", "45000") \
#     .option("kafka.request.timeout.ms", "60000") \
#     .load()

# query = df_kafka_raw.selectExpr("CAST(value AS STRING)") \
#     .writeStream \
#     .format("delta") \
#     .outputMode("append") \
#     .option("checkpointLocation", checkpoint_path_final) \
#     .trigger(processingTime='2 seconds') \
#     .start(bronze_path)

# query.awaitTermination()

# df_check = spark.read.format("delta").load(bronze_path)
# print(f"🚀 KẾT QUẢ CUỐI CÙNG: Đã hốt được {df_check.count()} dòng vào kho Bronze!")






















import time

# 1. Định nghĩa hàm thực hiện một lần "hốt" dữ liệu
def run_streaming_batch():
    print(f"🔄 Đang kiểm tra Kafka lúc: {time.strftime('%H:%M:%S')}...")
    
    # Thiết lập luồng (giữ nguyên các option cũ của mày)
    df_kafka_raw = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", bootstrap_server) \
        .option("kafka.security.protocol", "SASL_SSL") \
        .option("kafka.sasl.mechanism", "PLAIN") \
        .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";') \
        .option("subscribe", "loan_transaction") \
        .option("startingOffsets", "earliest") \
        .load()

    # Ghi dữ liệu vào Bronze
    query = df_kafka_raw.selectExpr("CAST(value AS STRING)") \
        .writeStream \
        .format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", checkpoint_path_final) \
        .trigger(availableNow=True) \
        .start(bronze_path)

    query.awaitTermination()
    
    # Sau khi chạy xong 1 batch, kiểm tra số lượng dòng
    df_check = spark.read.format("delta").load(bronze_path)
    print(f"✅ Tổng số dòng hiện tại trong Bronze: {df_check.count()}")

# 2. Vòng lặp chạy vô tận để giả lập Real-time
try:
    while True:
        run_streaming_batch()
        print("😴 Nghỉ 30 giây trước khi kiểm tra tiếp... (Nhấn nút Stop để dừng)")
        time.sleep(30)
except KeyboardInterrupt:
    print("🛑 Đã dừng luồng nhận dữ liệu.")



In [0]:
from pyspark.sql.functions import col, when, expr, from_json
from pyspark.sql.types import StructType, StructField, StringType

# 1. Đọc dữ liệu từ Bronze (Lúc này mới chỉ có 1 cột 'value')
df_bronze = spark.read.format("delta").load(bronze_path)

# 2. ĐỊNH NGHĨA SCHEMA (Phải khớp với các cột mày đã bắn từ Producer)
# Tao liệt kê các cột quan trọng nhất mày đang dùng
json_schema = StructType([
    StructField("loan_status", StringType(), True),
    StructField("loan_amnt", StringType(), True),
    StructField("int_rate", StringType(), True),
    StructField("installment", StringType(), True),
    StructField("annual_inc", StringType(), True),
    StructField("dti", StringType(), True),
    StructField("fico_range_low", StringType(), True),
    StructField("term", StringType(), True),
    StructField("grade", StringType(), True),
    StructField("emp_length", StringType(), True),
    StructField("home_ownership", StringType(), True)
])

# 3. GIẢI MÃ JSON (Bước này cực kỳ quan trọng để lòi ra cột loan_status)
df_parsed = df_bronze.withColumn("jsonData", from_json(col("value"), json_schema)) \
                     .select("jsonData.*")

# 4. Lọc và Gán nhãn (Bây giờ cột loan_status đã tồn tại)
df_silver = df_parsed.filter(col("loan_status").isin(["Fully Paid", "Charged Off"])) \
    .withColumn("label", when(col("loan_status") == "Fully Paid", 0.0).otherwise(1.0))

# 5. Safe Casting bằng try_cast cho các cột số
numeric_cols = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti", "fico_range_low"]
for c in numeric_cols:
    df_silver = df_silver.withColumn(c, expr(f"try_cast({c} as double)"))

# 6. Xử lý giá trị lỗi/thiếu
df_silver = df_silver.dropna(subset=["loan_amnt", "annual_inc"])
df_silver = df_silver.na.fill(0.0, subset=numeric_cols).na.fill("Unknown")

# 7. Lưu vào Silver
silver_path = "/Volumes/workspace/default/data_csv/loan_silver"
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

print(f"✅ Tầng Silver hoàn tất! Đã xử lý thành công {df_silver.count()} dòng.")



In [0]:

from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

df_silver_load = spark.read.format("delta").load(silver_path)

total_count = df_silver_load.count()
bad_count = df_silver_load.filter(col("label") == 1).count()

weight_ratio = (total_count - bad_count) / bad_count

df_gold_raw = df_silver_load.withColumn(
    "classWeight",
    when(col("label") == 1, weight_ratio).otherwise(1.0)
)

categorical_cols = ["term", "grade", "emp_length", "home_ownership"]

indexers = [
    StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
    for c in categorical_cols
]

assembler_inputs = numeric_cols + [c + "_idx" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="unscaled_features",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="unscaled_features",
    outputCol="features",
    withStd=True,
    withMean=False
)

pipeline = Pipeline(stages=indexers + [assembler] + [scaler])

sample_for_fit = df_gold_raw.limit(1000)
pipeline_model = pipeline.fit(sample_for_fit)

data_final = pipeline_model.transform(df_gold_raw)

gold_path = "/Volumes/workspace/default/data_csv/loan_gold"

data_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_path)

del pipeline_model

print(f"✅ Thành công! Đã tạo xong tầng Gold. Trọng số rủi ro: {weight_ratio:.2f}")




In [0]:



import os
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

ml_temp_path = "/Volumes/workspace/default/data_csv/ml_temp"
dbutils.fs.mkdirs(ml_temp_path)

os.environ['SPARKML_TEMP_DFS_PATH'] = ml_temp_path

train_data, test_data = spark.read.format("delta") \
    .load(gold_path) \
    .randomSplit([0.8, 0.2], seed=42)

rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50)
rf_model = rf.fit(train_data)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight"
)

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.1, 0.01]) \
    .build()

cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=BinaryClassificationEvaluator(),
    numFolds=3
)

cv_model = cv.fit(train_data)

print("✅ Đã huấn luyện xong Random Forest và Logistic Regression.")



In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def print_metrics(predictions, model_name):
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction"
    )

    acc = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
    prec = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})
    rec = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})
    auc = BinaryClassificationEvaluator().evaluate(predictions)

    print(f"📊 CHỈ SỐ CỦA {model_name}:")
    print(f"- Accuracy: {acc:.4f}")
    print(f"- Precision: {prec:.4f}")
    print(f"- Recall: {rec:.4f}")
    print(f"- AUC: {auc:.4f}\n")

rf_preds = rf_model.transform(test_data)
lr_preds = cv_model.bestModel.transform(test_data)

print_metrics(rf_preds, "RANDOM FOREST")
print_metrics(lr_preds, "LOGISTIC REGRESSION (BEST)")

output_redis_sim_path = "/Volumes/workspace/default/data_csv/redis_sim"

lr_preds.select("loan_amnt", "prediction", "label") \
    .write.format("delta") \
    .mode("overwrite") \
    .save(output_redis_sim_path)

# Lưu bảng kết quả dự đoán (đã có cột prediction)
lr_preds.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/data_csv/loan_results")
print("✅ Đã lưu kết quả dự đoán Loan.")




